In [48]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()

# Get MySQL connection details from environment variables
db_host = os.getenv('MYSQL_HOST')
db_port = os.getenv('MYSQL_PORT')
db_name_consumption = os.getenv('CONSUMPTION_DATABASE')
db_name_mart = os.getenv('MARTHOUSE_DATABASE')
db_user = os.getenv('MYSQL_USERNAME')
db_password = os.getenv('MYSQL_PASSWORD')

engine_consumption = create_engine(
    f"mysql+mysqlconnector://{db_user}:{db_password}@{db_host}:{db_port}/{db_name_consumption}"
)

engine_mart = create_engine(
    f"mysql+mysqlconnector://{db_user}:{db_password}@{db_host}:{db_port}/{db_name_mart}"
)

def load_table(query, engine):
    return pd.read_sql(query, engine)

In [49]:
top_cities = load_table("SELECT * FROM consumption_top10_polluting_cities", engine_consumption)
stations = load_table("SELECT * FROM dim_station", engine_mart)
hourly = load_table("SELECT * FROM consumption_city_hourly_pollution", engine_consumption)
stations_summary = load_table("SELECT * FROM consumption_city_station_coverage", engine_consumption)

In [50]:
import plotly.express as px

# Compute centroid per city
city_geo = stations.groupby("city_name").agg({
    "latitude": "mean",
    "longitude": "mean"
}).reset_index()

top_cities_geo = top_cities.merge(city_geo, on="city_name")

fig = px.scatter_map(
    top_cities_geo,
    lat="latitude",
    lon="longitude",
    color="pollutant_type",
    size="avg_pollution",
    hover_name="city_name",
    zoom=7,
    title="Top 10 Polluted Cities in Flanders",
    height=600
)
fig.write_html("polluted_cities_flanders.html")
fig.show()

In [30]:
fig = px.bar(
    top_cities,
    x="avg_pollution",
    y="city_name",
    color="pollutant_type",
    orientation="h",
    title="Top 10 Most Polluted Cities",
    hover_data=["pollutant_type"]
)
fig.write_html("polluted_cities_flanders_bar.html")
fig.show()

In [8]:
EU_THRESHOLDS = {
    "PM10": 50,
    "PM2.5": 25,
    "NO2": 40,
    "SO2": 20
}
fig = px.bar(
    top_cities,
    x="avg_pollution",
    y="city_name",
    color="pollutant_type",
    orientation="h",
    title="Top Cities vs EU Thresholds"
)

# Add threshold lines
for pollutant, value in EU_THRESHOLDS.items():
    fig.add_vline(
        x=value,
        line_dash="dash",
        annotation_text=f"{pollutant} limit",
        annotation_position="top"
    )

fig.show()

In [33]:
import plotly.express as px

# We use facet_col_wrap for a clean 2x2 dashboard look
fig1 = px.bar(
    top_cities,
    x="avg_pollution",
    y="city_name",
    color="avg_pollution",
    orientation="h",
    facet_col="pollutant_type",
    facet_col_wrap=2,
    title="<b>Top 10 Polluted Cities: Multi-Pollutant Analysis</b>",
    labels={"avg_pollution": "Avg Concentration", "city_name": ""},
    height=800
)

# Crucial: Allow each subplot to show its own unique top 10 list
fig1.update_yaxes(matches=None, showticklabels=True, categoryorder="total ascending")
fig1.update_layout(coloraxis_showscale=False, title_x=0.5)
fig1.write_html("polluted_cities_flanders_bar2.html")
fig1.show()

In [51]:
import plotly.express as px
import pandas as pd

city_query = "Vilvoorde"
target_date = "2026-01-03" 

# 1. Filter data
df_city = hourly[
    (hourly["city_name"] == city_query) & 
    (hourly["measured_at_date"].astype(str) == target_date)
].copy()

# 2. Identify the peak for each pollutant
# We find the max value per pollutant and create a label only for those rows
df_city['is_peak'] = df_city.groupby('pollutant_type')['avg_hourly_pollution'].transform(max) == df_city['avg_hourly_pollution']

# Create the label column: only populate it if it's a peak, otherwise leave it empty
df_city["peak_label"] = df_city.apply(
    lambda row: f"{row['avg_hourly_pollution']:.1f}" if row['is_peak'] else "", 
    axis=1
)

# 3. Create the figure
fig = px.line(
    df_city,
    x="hour",
    y="avg_hourly_pollution",
    color="pollutant_type",
    markers=True,
    text="peak_label",  # Only the peak values will show text
    hover_data=["measured_at_date"],
    title=f"Hourly Pollution Pattern with Peaks - {city_query} ({target_date})"
)

# 4. Styling the labels
fig.update_traces(
    textposition="top center",
    textfont=dict(family="Arial", size=12, color="black")
)

# Improve layout to ensure labels aren't cut off at the top
fig.update_layout(yaxis=dict(range=[0, df_city['avg_hourly_pollution'].max() * 1.15]))
fig.write_html(f"{city_query}_hourly_pollution_{target_date}.html")
fig.show()

In [18]:
fig = px.scatter_map(
    stations,
    lat="latitude",
    lon="longitude",
    hover_name="station_name",
    color="city_name",
    zoom=7,
    title="Monitoring Stations Distribution",
    height=600
)
fig.write_html("monitoring_stations_distribution.html")
fig.show()

In [12]:
fig = px.bar(
    stations_summary,
    x="city_name",
    y="nb_stations",
    title="Number of Stations per City"
)

fig.show()

In [14]:
stations_summary = load_table("SELECT * FROM consumption_city_station_coverage", engine_consumption)

merged = top_cities.merge(stations_summary, on="city_name")

import plotly.express as px

fig = px.scatter(
    merged,
    x="nb_stations",
    y="avg_pollution",
    color="pollutant_type",
    hover_name="city_name",
    size="avg_pollution",
    title="Pollution vs Number of Stations (Interactive)"
)

fig.show()

In [18]:
import plotly.graph_objects as go

merged = top_cities.merge(
    stations_summary,
    on="city_name",
    how="inner"
)

pollutants = merged["pollutant_type"].unique()

fig = go.Figure()

for p in pollutants:
    df = merged[merged["pollutant_type"] == p]

    fig.add_trace(go.Scatter(
        x=df["nb_stations"],
        y=df["avg_pollution"],
        mode="markers+text",  # 👈 add text
        text=df["city_name"],  # 👈 city names displayed
        textposition="top center",
        name=p,
        visible=(p == pollutants[0]),
        marker=dict(size=12, opacity=0.7)
    ))

buttons = []

for i, p in enumerate(pollutants):
    visibility = [False] * len(pollutants)
    visibility[i] = True

    buttons.append(dict(
        label=p,
        method="update",
        args=[
            {"visible": visibility},
            {"title": f"Pollution vs Stations — {p}"}
        ]
    ))

fig.update_layout(
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        showactive=True
    )],
    title=f"Pollution vs Stations — {pollutants[0]}",
    xaxis_title="Number of Stations",
    yaxis_title="Average Pollution",
    height=600
)

fig.show()

In [24]:
import plotly.express as px
import pandas as pd

# 1. Load data
merged_mart = load_table("SELECT * FROM consumption_city_avg", engine_consumption)

# 2. Create the figure - CRITICAL: Add hover_data to populate customdata
cities = sorted(merged_mart["city_name"].unique())
pollutants = sorted(merged_mart["pollutant_type"].unique())

fig = px.scatter_map(
    merged_mart,
    lat="latitude",
    lon="longitude",
    color="pollutant_type",
    size="avg_pollution",
    hover_name="station_name",
    # hover_data creates the 'customdata' array we use for filtering
    hover_data=["city_name", "pollutant_type"], 
    zoom=7
)

# 3. Create City Buttons
city_buttons = []
city_buttons.append(dict(
    method="update",
    label="All Cities",
    args=[{"visible": [True] * len(fig.data)}, {"title": "All Cities Monitoring"}]
))

for city in cities:
    visibility = []
    for trace in fig.data:
        # customdata[0][0] corresponds to 'city_name' because it's first in hover_data
        if trace.customdata is not None and trace.customdata[0][0] == city:
            visibility.append(True)
        else:
            visibility.append(False)

    city_buttons.append(dict(
        method="update",
        label=city,
        args=[{"visible": visibility}, {"title": f"Monitoring in {city}"}]
    ))

# 4. Create Pollutant Buttons
pollutant_buttons = []
pollutant_buttons.append(dict(
    method="update",
    label="All Pollutants",
    args=[{"visible": [True] * len(fig.data)}, {"title": "All Pollutants"}]
))

for poll in pollutants:
    # trace.name matches the 'pollutant_type' because we used it as the 'color' argument
    pollutant_buttons.append(dict(
        method="update",
        label=poll,
        args=[{"visible": [poll == t.name for t in fig.data]},
              {"title": f"Focus on {poll}"}]
    ))

# 5. Add menus and Annotations (for the labels)
fig.update_layout(
    updatemenus=[
        dict(
            buttons=city_buttons,
            direction="down",
            showactive=True,
            x=0.05, y=1.12
        ),
        dict(
            buttons=pollutant_buttons,
            direction="down",
            showactive=True,
            x=0.25, y=1.12
        )
    ],
    # Adding titles for the dropdowns via annotations
    annotations=[
        dict(text="City:", x=0.05, y=1.18, xref="paper", yref="paper", showarrow=False),
        dict(text="Pollutant:", x=0.25, y=1.18, xref="paper", yref="paper", showarrow=False)
    ],
    map_style="open-street-map",
    margin={"r":0,"t":50,"l":0,"b":0}
)

fig.show()

In [54]:
import plotly.express as px

city_query = "Steenokkerzeel"
# 1. Filter for your target city
city_df = df[df['city_name'] == city_query].sort_values(["pollutant_type", "station_avg_pollution"])

# 2. Create the Bar Chart
fig = px.bar(
    city_df,
    x="station_avg_pollution",
    y="station_name",
    color="pollutant_type",  # This ensures each type has its own visible color
    orientation="h",
    text="city_station_count", 
    title=f"<b>Station Air Quality Profile: {city_df['city_name'].iloc[0]}</b>",
    labels={
        "station_avg_pollution": "Avg Concentration (µg/m³)",
        "station_name": "Monitoring Station",
        "pollutant_type": "Pollutant Type",
        "city_station_count": "Sensor Count"
    },
    # barmode="group" ensures that if a station has multiple pollutants, 
    # they show up as separate bars next to each other.
    barmode="group",
    height=max(400, len(city_df) * 30) # Dynamic height based on number of stations
)

# 3. Clean up the text and Legend
fig.update_traces(
    texttemplate='<b>%{text} Sensors</b>', 
    textposition='outside',
    cliponaxis=False
)

fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor='lightgrey'),
    # Move the Legend to the top and make it horizontal
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        title_text="" # Removes the 'pollutant_type' header to save space
    ),
    margin=dict(l=150, r=120, t=100, b=50) 
)
fig.write_html(f"Air_Quality_Profile_{city_query}.html")
fig.show()

In [43]:
import plotly.express as px

# 1. Group pollutants by station so we have one row per station
# This creates a string like "NO2, PM10" for the tooltip
stations_aggregated = df.groupby(
    ['station_name', 'city_name', 'latitude', 'longitude']
)['pollutant_type'].apply(lambda x: ', '.join(sorted(x.unique()))).reset_index()

# 2. Create the Map
fig = px.scatter_map(
    stations_aggregated,
    lat="latitude",
    lon="longitude",
    hover_name="station_name",
    # We keep city_name as color to see the distribution
    color="city_name", 
    # Add the combined pollutant list to the hover data
    hover_data={
        "city_name": True,
        "pollutant_type": True, # This now shows the list of pollutants
        "latitude": False,
        "longitude": False
    },
    zoom=7,
    title="<b>Monitoring Network: Station Locations & Measured Pollutants</b>",
    height=600
)

# 3. Enhance Visuals
fig.update_traces(marker={'size': 12}) # Make dots a bit more visible
fig.update_layout(
    map_style="open-street-map",
    showlegend=False, # Optional: hide legend if there are too many cities
    margin={"r":0,"t":50,"l":0,"b":0}
)

fig.write_html("monitoring_stations_distribution.html")
fig.show()